In [1]:
# ===== Step 8.0: review the Step 7 rerun (reads saved tables only) =====
import glob, os, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
hit = glob.glob("/kaggle/input/**/figures/table_kd_main.csv", recursive=True)
assert hit, "table_kd_main.csv not found - is wildleaf-step7 attached, and did the commit finish?"
FIG = os.path.dirname(hit[0]); ROOT = os.path.dirname(FIG); print("dataset root:", ROOT, "|", os.listdir(ROOT))
for name in ["table_kd_main", "table_kd_significance", "table_kd_ablations", "table_kd_expert_calibration",
             "table_cross_dataset"]:
    f = f"{FIG}/{name}.csv"
    print(f"\n=== {name} ===")
    print(pd.read_csv(f).round(4).to_string(index=False) if os.path.exists(f) else "MISSING")
f = f"{FIG}/table_kd_per_class.csv"
if os.path.exists(f):
    PC = pd.read_csv(f)
    print("\n=== per-class: improved / total ===\n",
          PC.groupby(["data", "healthy"]).gain_pp.agg(classes="size", mean_gain="mean", improved=lambda g: (g > 0).sum()).round(2))
    for d in ["pd", "pw"]:
        s = PC[PC.data == d].sort_values("gain_pp")
        print(d, "worst 3:", list(zip(s.class_name.head(3), s.gain_pp.head(3).round(1))))
want = ["fig7_1_kd_main", "fig7_2_kd_ablations", "fig7_3_per_class_gain", "fig7_4_calibration", "fig7_5_cross_dataset",
        "figD11_fg_masks", "figD12_pca_components", "figD13_mask_failures", "figD14_attn_vs_pca_masks"]
print("\n=== figures ===")
for w in want: print(f"{w:28s}", "OK" if glob.glob(f"{FIG}/{w}.png") else "MISSING")
ck = glob.glob(f"{ROOT}/students/ckpt/*.pt"); print("\ncheckpoints saved:", len(ck))

dataset root: /kaggle/input/datasets/mashfii/wildleaf-step7 | ['students', 'train_student.py', 'cache', '__huggingface_repos__.json', 'figures', 'wildleaf_step7', 'feats']

=== table_kd_main ===
data                  model  n  baseline  base_sd     kd  kd_sd  delta_pp  delta_sd wins
  pd        efficientnet_b0  5    0.6233   0.0346 0.6619 0.0340    3.8522    2.6215  4/5
  pd  mobilenetv3_large_100  5    0.6130   0.0195 0.6695 0.0409    5.6494    3.4881  5/5
  pd mobilenetv4_conv_small  5    0.5173   0.0244 0.6094 0.0294    9.2053    2.4150  5/5
  pw        efficientnet_b0  3    0.6663   0.0066 0.7153 0.0058    4.9003    1.0129  3/3
  pw  mobilenetv3_large_100  3    0.6610   0.0058 0.7076 0.0032    4.6653    0.8163  3/3
  pw mobilenetv4_conv_small  1    0.5764      NaN 0.6561    NaN    7.9642       NaN  1/1

=== table_kd_significance ===
data                  model  n  mean_gain_pp  ci95_low  ci95_high    t_p  wilcoxon_p
  pd        efficientnet_b0  5        3.8522    0.5972     7.1072 

In [2]:
# ===== Step 8 notebook setup: restore from wildleaf-step7, link cache, cache weights =====
import os, glob, shutil, timm
from transformers import AutoModel
W = "/kaggle/working"
ROOT = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/figures/table_kd_main.csv", recursive=True)[0]))
for d in ["feats", "figures", "students"]:
    shutil.copytree(f"{ROOT}/{d}", f"{W}/{d}", dirs_exist_ok=True); print(d, "restored")
shutil.copy(f"{ROOT}/train_student.py", f"{W}/train_student.py")
os.makedirs(f"{W}/cache", exist_ok=True)
for t in ["pd", "pw"]:
    dst = f"{W}/cache/{t}_256.npy"
    if not os.path.exists(dst): os.symlink(f"{ROOT}/cache/{t}_256.npy", dst)
print("cache linked:", os.listdir(f"{W}/cache"))
for m in ["mobilenetv3_large_100.ra_in1k", "efficientnet_b0.ra_in1k",
          "mobilenetv4_conv_small.e2400_r224_in1k", "resnet50.a1_in1k"]:
    timm.create_model(m, pretrained=True); print("ok", m)
os.environ["HF_HUB_OFFLINE"] = "1"; print("Setup DONE (offline mode ON).")

feats restored
figures restored
students restored
cache linked: ['pw_256.npy', 'pd_256.npy']


model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

ok mobilenetv3_large_100.ra_in1k


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

ok efficientnet_b0.ra_in1k


model.safetensors:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

ok mobilenetv4_conv_small.e2400_r224_in1k


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

ok resnet50.a1_in1k
Setup DONE (offline mode ON).


In [4]:
# ===== Step 7.7: cross-dataset (PW->PD, PD->PW) baseline vs KD =====
import os, time, subprocess, numpy as np, pandas as pd, torch, torch.nn.functional as F, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
W, FEAT, FIG, STU = "/kaggle/working", "/kaggle/working/feats", "/kaggle/working/figures", "/kaggle/working/students"; dev = "cuda:0"
PD = pd.read_csv(f"{FEAT}/pd_manifest.csv", low_memory=False); PW = pd.read_csv(f"{FEAT}/pw_manifest.csv", low_memory=False)
truthy = lambda s: s.astype(str).str.lower().isin(["true", "1", "1.0", "test"])
print("PD xd_pd2pw_role:", PD.xd_pd2pw_role.value_counts(dropna=False).to_dict())
print("PD xd_pw2pd_test_clean:", PD.xd_pw2pd_test_clean.value_counts(dropna=False).to_dict())
print("PW xd_pw2pd_role:", PW.xd_pw2pd_role.value_counts(dropna=False).to_dict())
print("PW xd_pd2pw_test:", PW.xd_pd2pw_test.value_counts(dropna=False).to_dict())

# ---------- 1. shared label space = PlantDoc class ids ----------
pd_id = PD.drop_duplicates("class_name").set_index("class_name").class_id
pw_used = PW.xd_pw2pd_role.isin(["train", "val"]) | truthy(PW.xd_pd2pw_test)
unk = set(PW.loc[pw_used, "xd_class"].dropna()) - set(pd_id.index)
assert not unk, f"xd_class values not in PlantDoc classes: {list(unk)[:5]}"
pd_used = PD.xd_pd2pw_role.isin(["train", "val"]) | (PD.p2_role == "test")
A = PW[pw_used].copy(); A["src"] = "pw"; A["row_src"] = np.where(pw_used)[0]; A["class_id"] = A.xd_class.map(pd_id).astype(int)
B = PD[pd_used].copy(); B["src"] = "pd"; B["row_src"] = np.where(pd_used)[0]
A["pw2pd_role"] = A.xd_pw2pd_role.where(A.xd_pw2pd_role.isin(["train", "val"]), "excluded")
A["pd2pw_role"] = np.where(truthy(A.xd_pd2pw_test), "test", "excluded")
B["pw2pd_role"] = np.where(B.p2_role == "test", "test", "excluded")
B["pd2pw_role"] = B.xd_pd2pw_role.where(B.xd_pd2pw_role.isin(["train", "val"]), "excluded")
cross = pd.read_csv([p for p in __import__("glob").glob("/kaggle/input/**/pd_pw_cross_duplicates.csv", recursive=True)][0])
twin_pd = set(cross.loc[cross.dup.astype(str).str.lower() == "true", "pd_uid"])
B["xd_twin"] = (B.p2_role == "test") & ~truthy(B.xd_pw2pd_test_clean); A["xd_twin"] = False
assert int(((B.p2_role == "test") & truthy(B.xd_pw2pd_test_clean)).sum()) == 173, "twin-free subset is not inside the clean test"
X = pd.concat([A, B], ignore_index=True)
X["dup_group"] = X.src + "_" + X.dup_group.astype(str)
cnt = {k: X[k].value_counts().to_dict() for k in ["pw2pd_role", "pd2pw_role"]}; print("\nxd roles:", cnt)
exp = {"pw2pd_role": {"train": 3727, "val": 527, "test": 220}, "pd2pw_role": {"train": 1798, "val": 306, "test": 1111}}
for k, e in exp.items():
    for r, n in e.items(): assert cnt[k].get(r) == n, f"{k}/{r}: got {cnt[k].get(r)}, audit says {n}"
print("counts match audit | PD test twin-free:", int(((X.pw2pd_role == "test") & ~X.xd_twin).sum()))
ORGAN = ["Apple Scab Leaf", "Apple rust leaf", "Tomato leaf late blight", "grape leaf black rot"]
X["organ_shift"] = X.class_id.isin([pd_id[c] for c in ORGAN if c in pd_id])
X.to_csv(f"{FEAT}/xd_manifest.csv", index=False)

# ---------- 2. xd image cache (indexed from existing caches) ----------
C = {t: np.load(f"{W}/cache/{t}_256.npy", mmap_mode="r") for t in ["pd", "pw"]}
arr = np.lib.format.open_memmap(f"{W}/cache/xd_256.npy", mode="w+", dtype=np.uint8, shape=(len(X), 256, 256, 3))
for t in ["pw", "pd"]:
    rows = np.where(X.src.values == t)[0]; srcr = X.row_src.values[rows]
    for i in range(0, len(rows), 512): arr[rows[i:i+512]] = C[t][srcr[i:i+512]]
arr.flush(); del arr; print("xd cache:", len(X), "images")

# ---------- 3. cross-fitted KD targets on each source ----------
l2 = lambda a: a / np.linalg.norm(a, axis=1, keepdims=True)
def feats(t):
    L = lambda k: np.load(f"{FEAT}/{t}_{k}.npy")
    return np.hstack([l2(np.hstack([L("dino_cls"), L("dino_mean")])), l2(L("clip"))]).astype(np.float32)
FX = np.zeros((len(X), 2048), np.float32)
for t in ["pw", "pd"]:
    rows = np.where(X.src.values == t)[0]; FX[rows] = feats(t)[X.row_src.values[rows]]
WD = pd.read_csv(f"{FEAT}/teacher_probe_results.csv").query("protocol=='p2_role' and feat=='dino+clip'").set_index("dataset").wd
def fit_logits(Xtr, ytr, Xev, wd, K=27):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Am = torch.tensor((Xtr - mu) / sd, device=dev); yy = torch.tensor(ytr, device=dev)
    Wt = torch.zeros(Am.shape[1], K, device=dev, requires_grad=True); b = torch.zeros(K, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([Wt, b], lr=1, max_iter=300, line_search_fn="strong_wolfe")
    def c():
        opt.zero_grad(); l = F.cross_entropy(Am @ Wt + b, yy) + wd * (Wt**2).sum(); l.backward(); return l
    opt.step(c); return (torch.tensor((Xev - mu) / sd, device=dev) @ Wt.detach() + b.detach()).cpu().numpy()
def fit_T(z, yy):
    z = torch.tensor(z, device=dev); yy = torch.tensor(yy, device=dev); lt = torch.zeros(1, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([lt], lr=.1, max_iter=100)
    def c():
        opt.zero_grad(); l = F.cross_entropy(z / lt.exp(), yy); l.backward(); return l
    opt.step(c); return float(lt.exp())
y = X.class_id.values; summ = pd.read_csv(f"{FEAT}/kd_targets/summary.csv")
summ = summ[summ.dataset != "xd"]; teacher_rows = []
for pc, src in [("pw2pd_role", "pw"), ("pd2pw_role", "pd")]:
    r = X[pc].values; tr = np.where(r == "train")[0]; ev = np.where(np.isin(r, ["val", "test"]))[0]
    va, te = np.where(r == "val")[0], np.where(r == "test")[0]; Z = np.zeros((len(X), 27), np.float32)
    for a_, b_ in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(FX[tr], y[tr], X.dup_group.values[tr]):
        Z[tr[b_]] = fit_logits(FX[tr[a_]], y[tr[a_]], FX[tr[b_]], WD[src])
    Z[ev] = fit_logits(FX[tr], y[tr], FX[ev], WD[src]); T = fit_T(Z[va], y[va])
    np.save(f"{FEAT}/kd_targets/xd_{pc.replace('_role','')}_logits.npy", Z.astype(np.float16))
    summ = pd.concat([summ, pd.DataFrame([dict(dataset="xd", protocol=pc, n_train=len(tr), T_val=T)])])
    tf = ~X.xd_twin.values[te]
    teacher_rows.append(dict(direction=pc.replace("_role", ""), method="teacher", model="DINOv2+CLIP", seed=0,
                             f1_all=f1_score(y[te], Z[te].argmax(1), average="macro"),
                             f1_twinfree=f1_score(y[te][tf], Z[te][tf].argmax(1), average="macro")))
    print(f"{pc}: teacher cross-dataset test F1={teacher_rows[-1]['f1_all']:.4f}  T={T:.3f}")
summ.to_csv(f"{FEAT}/kd_targets/summary.csv", index=False)

# ---------- 4. runs ----------
def run_pair(jobs):
    procs = []
    for gpu, args in enumerate(jobs):
        tag = "_".join(x for x in args if not x.startswith("--"))[:110]
        log = open(f"{STU}/log_{tag}.txt", "w")
        procs.append((subprocess.Popen(["python", f"{W}/train_student.py", *args],
                      env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}, stdout=log, stderr=subprocess.STDOUT), tag))
    for p, tag in procs:
        p.wait()
        if p.returncode != 0: print("FAILED", tag, open(f"{STU}/log_{tag}.txt").read()[-2000:])
MODELS = {"mobilenetv3_large_100": "mobilenetv3_large_100.ra_in1k", "efficientnet_b0": "efficientnet_b0.ra_in1k"}
jobs = [["--data", "xd", "--proto", pc, "--model", m, "--seed", str(s), "--lr", "1e-3", "--epochs", "30",
         "--kd_alpha", al, "--kd_tau", "4"] for pc in ["pw2pd_role", "pd2pw_role"] for m in MODELS.values()
        for s in [0, 1] for al in ["0", "0.5"]]
t0 = time.time()
for i in range(0, len(jobs), 2):
    run_pair(jobs[i:i+2]); print(f"pair {i//2+1}/{len(jobs)//2} done ({(time.time()-t0)/60:.1f} min)")

# ---------- 5. analysis ----------
rows = list(teacher_rows)
for pc in ["pw2pd", "pd2pw"]:
    for short in MODELS:
        for s in [0, 1]:
            for meth, suf in [("baseline", ""), ("kd", "_kd0.5t4")]:
                tag = f"xd_{pc}_{short}_s{s}_lr0.001{suf}"
                if not os.path.exists(f"{STU}/logits/{tag}_test.npy"): print("missing", tag); continue
                Lg = np.load(f"{STU}/logits/{tag}_test.npy").astype(np.float32); idx = pd.read_csv(f"{STU}/logits/{tag}_test_uids.csv").row.values
                yt, yp = y[idx], Lg.argmax(1); tf = ~X.xd_twin.values[idx]; org = X.organ_shift.values[idx]
                rows.append(dict(direction=pc, method=meth, model=short, seed=s, f1_all=f1_score(yt, yp, average="macro"),
                                 f1_twinfree=f1_score(yt[tf], yp[tf], average="macro"),
                                 acc_organ=(yp[org] == yt[org]).mean(), acc_other=(yp[~org] == yt[~org]).mean()))
XD = pd.DataFrame(rows); XD.to_csv(f"{FIG}/table_cross_dataset_runs.csv", index=False)
S = XD.groupby(["direction", "model", "method"]).agg(runs=("f1_all", "size"), f1_all=("f1_all", "mean"), f1_twinfree=("f1_twinfree", "mean"),
                                                     acc_organ_shift=("acc_organ", "mean"), acc_other=("acc_other", "mean")).round(4)
pd.set_option("display.width", 200); print("\n=== CROSS-DATASET (27 shared classes) ===\n", S.to_string())
S.to_csv(f"{FIG}/table_cross_dataset.csv")
# in-domain reference (same test set, trained in-domain): PD P2 teacher / students printed for context
fig, axs = plt.subplots(1, 2, figsize=(9, 3.3), sharey=True)
NICE = {"mobilenetv3_large_100": "MobileNetV3-L", "efficientnet_b0": "EfficientNet-B0"}
for ax, pc, title in zip(axs, ["pw2pd", "pd2pw"], ["Train PlantWild → test PlantDoc (twin-free)", "Train PlantDoc → test PlantWild"]):
    s = S.loc[pc]; x = np.arange(len(MODELS))
    b = [s.loc[(m, "baseline"), "f1_twinfree"] for m in MODELS]; k = [s.loc[(m, "kd"), "f1_twinfree"] for m in MODELS]
    ax.bar(x - .2, b, .4, color="#8C8C8C", label="Baseline"); ax.bar(x + .2, k, .4, color="#4C72B0", label="KD")
    for i in range(len(x)): ax.text(x[i] + .2, k[i] + .01, f"{100*(k[i]-b[i]):+.1f} pp", ha="center", fontsize=8)
    ax.axhline(s.loc[("DINOv2+CLIP", "teacher"), "f1_twinfree"], color="#C44E52", ls="--", label="Teacher")
    ax.set_xticks(x); ax.set_xticklabels([NICE[m] for m in MODELS]); ax.set_title(title, fontsize=9)
axs[0].set_ylabel("Test macro-F1 (27 shared classes)"); axs[0].legend(fontsize=8)
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/fig7_5_cross_dataset.{e}", dpi=300, bbox_inches="tight")
plt.close(fig)
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG); log = log[log.file != "fig7_5_cross_dataset"]
pd.concat([log, pd.DataFrame([dict(file="fig7_5_cross_dataset", step="7",
    caption="Cross-dataset generalisation on 27 shared classes: baseline vs KD students and the teacher (mean of 2 seeds).")])]).to_csv(LOG, index=False)
print("saved fig7_5_cross_dataset\nStep 7.7 DONE.")

PD xd_pd2pw_role: {'train': 1798, 'excluded': 472, 'val': 306}
PD xd_pw2pd_test_clean: {0: 2403, 1: 173}
PW xd_pw2pd_role: {'excluded': 14288, 'train': 3727, 'val': 527}
PW xd_pd2pw_test: {0: 17431, 1: 1111}

xd roles: {'pw2pd_role': {'train': 3727, 'excluded': 3215, 'val': 527, 'test': 220}, 'pd2pw_role': {'excluded': 4474, 'train': 1798, 'test': 1111, 'val': 306}}
counts match audit | PD test twin-free: 173
xd cache: 7689 images


/tmp/ipykernel_58/704734799.py:70: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  opt.step(c); return float(lt.exp())


pw2pd_role: teacher cross-dataset test F1=0.7265  T=2.043
pd2pw_role: teacher cross-dataset test F1=0.6278  T=1.086
pair 1/8 done (4.5 min)
pair 2/8 done (8.9 min)
pair 3/8 done (15.6 min)
pair 4/8 done (22.4 min)
pair 5/8 done (25.4 min)
pair 6/8 done (28.5 min)
pair 7/8 done (32.8 min)
pair 8/8 done (37.0 min)

=== CROSS-DATASET (27 shared classes) ===
                                           runs  f1_all  f1_twinfree  acc_organ_shift  acc_other
direction model                 method                                                         
pd2pw     DINOv2+CLIP           teacher      1  0.6278       0.6278              NaN        NaN
          efficientnet_b0       baseline     2  0.4233       0.4233           0.3789     0.4537
                                kd           2  0.4856       0.4856           0.4923     0.4984
          mobilenetv3_large_100 baseline     2  0.4249       0.4249           0.4124     0.4482
                                kd           2  0.4641       0.464

In [6]:
# ===== Step 8.1: ONNX export + INT8 static quantisation + CPU latency (PW P2, seed 0) =====
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)
import os, time, glob, numpy as np, pandas as pd, torch, torch.nn.functional as F, timm, onnxruntime as ort, matplotlib.pyplot as plt
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
from sklearn.metrics import f1_score
W, FEAT, FIG, STU = "/kaggle/working", "/kaggle/working/feats", "/kaggle/working/figures", "/kaggle/working/students"
OUT = f"{W}/deploy"; os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(f"{FEAT}/pw_manifest.csv", low_memory=False); y_all = df.class_id.values; K = int(y_all.max()) + 1
ARR = np.load(f"{W}/cache/pw_256.npy", mmap_mode="r")
te = np.where(df.p2_role.values == "test")[0]; tr = np.where(df.p2_role.values == "train")[0]
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def prep(idx):                                    # identical to training-time eval preprocessing
    x = torch.from_numpy(np.ascontiguousarray(ARR[idx])).permute(0, 3, 1, 2).float() / 255.
    x = (F.interpolate(x, size=224, mode="bicubic", antialias=True, align_corners=False).clamp(0, 1) - MEAN) / STD
    return x.numpy().astype(np.float32)

class Calib(CalibrationDataReader):
    def __init__(s, name):
        ids = np.sort(np.random.default_rng(0).choice(tr, 256, replace=False))
        s.batches = iter([{name: prep(ids[i:i+32])} for i in range(0, 256, 32)])
    def get_next(s): return next(s.batches, None)

def evaluate(path):
    so = ort.SessionOptions(); so.intra_op_num_threads = 4
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"]); name = sess.get_inputs()[0].name
    pred = np.concatenate([sess.run(None, {name: prep(te[i:i+64])})[0].argmax(1) for i in range(0, len(te), 64)])
    return f1_score(y_all[te], pred, average="macro"), (pred == y_all[te]).mean()

def latency(path, threads):
    so = ort.SessionOptions(); so.intra_op_num_threads = threads; so.inter_op_num_threads = 1
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"]); name = sess.get_inputs()[0].name
    x = prep(te[:1])
    for _ in range(20): sess.run(None, {name: x})
    ts = []
    for _ in range(100):
        t = time.perf_counter(); sess.run(None, {name: x}); ts.append(time.perf_counter() - t)
    return 1000 * np.median(ts)

MODELS = {"mobilenetv3_large_100": "MobileNetV3-L", "efficientnet_b0": "EfficientNet-B0", "mobilenetv4_conv_small": "MobileNetV4-S"}
TIMM = {"mobilenetv3_large_100": "mobilenetv3_large_100.ra_in1k", "efficientnet_b0": "efficientnet_b0.ra_in1k",
        "mobilenetv4_conv_small": "mobilenetv4_conv_small.e2400_r224_in1k"}
rows = []
for short, nice in MODELS.items():
    for meth, suf in [("baseline", ""), ("kd", "_kd0.5t4")]:
        ck = f"{STU}/ckpt/pw_p2_{short}_s0_lr0.001{suf}.pt"
        if not os.path.exists(ck): print("missing", ck); continue
        net = timm.create_model(TIMM[short], pretrained=False, num_classes=K)
        net.load_state_dict(torch.load(ck, map_location="cpu")); net.eval()
        f32 = f"{OUT}/{short}_{meth}_fp32.onnx"; i8 = f"{OUT}/{short}_{meth}_int8.onnx"
        torch.onnx.export(net, torch.randn(1, 3, 224, 224), f32, input_names=["input"], output_names=["logits"],
                          dynamic_axes={"input": {0: "b"}, "logits": {0: "b"}}, opset_version=17,dynamo=False)
        quantize_static(f32, i8, Calib("input"), quant_format=QuantFormat.QDQ, per_channel=True,
                        activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
        for prec, path in [("FP32", f32), ("INT8", i8)]:
            f1, acc = evaluate(path)
            rows.append(dict(model=nice, method=meth, precision=prec, size_MB=os.path.getsize(path) / 1e6,
                             test_f1=f1, test_acc=acc, cpu_ms_1thread=latency(path, 1), cpu_ms_4threads=latency(path, 4)))
            print(f"{nice:16s} {meth:8s} {prec}: F1={f1:.4f}  size={rows[-1]['size_MB']:.1f}MB  "
                  f"1T={rows[-1]['cpu_ms_1thread']:.1f}ms  4T={rows[-1]['cpu_ms_4threads']:.1f}ms", flush=True)

D = pd.DataFrame(rows).round(4); D.to_csv(f"{FIG}/table_int8_deployment.csv", index=False)
P = D.pivot_table(index=["model", "method"], columns="precision", values=["test_f1", "size_MB", "cpu_ms_1thread"])
P[("int8_drop_pp", "")] = 100 * (P[("test_f1", "INT8")] - P[("test_f1", "FP32")])
P[("speedup_1T", "")] = P[("cpu_ms_1thread", "FP32")] / P[("cpu_ms_1thread", "INT8")]
pd.set_option("display.width", 220); print("\n=== INT8 DEPLOYMENT (PlantWild P2 test, seed 0) ===\n", P.round(3).to_string())

fig, ax = plt.subplots(figsize=(6, 3.8)); mk = {"FP32": "o", "INT8": "^"}; col = {"baseline": "#8C8C8C", "kd": "#4C72B0"}
for _, r in D.iterrows():
    ax.scatter(r.cpu_ms_1thread, r.test_f1, marker=mk[r.precision], color=col[r.method], s=50)
    if r.precision == "INT8" and r.method == "kd": ax.annotate(r.model, (r.cpu_ms_1thread, r.test_f1), fontsize=7, xytext=(4, 3), textcoords="offset points")
for p, m in mk.items(): ax.scatter([], [], marker=m, color="gray", label=p)
for k, c in col.items(): ax.scatter([], [], marker="s", color=c, label=k)
ax.set_xlabel("CPU latency, batch 1, 1 thread (ms)"); ax.set_ylabel("Test macro-F1 (PlantWild P2)"); ax.legend(fontsize=8); ax.grid(alpha=.3)
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/fig8_1_int8_deployment.{e}", dpi=300, bbox_inches="tight")
plt.close(fig)
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG); log = log[log.file != "fig8_1_int8_deployment"]
pd.concat([log, pd.DataFrame([dict(file="fig8_1_int8_deployment", step="8",
    caption="Accuracy vs single-thread CPU latency for FP32 and INT8 (ONNX Runtime) baseline and KD students.")])]).to_csv(LOG, index=False)
print("saved fig8_1_int8_deployment\nStep 8.1 DONE.")

/tmp/ipykernel_58/361664559.py:51: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net, torch.randn(1, 3, 224, 224), f32, input_names=["input"], output_names=["logits"],


MobileNetV3-L    baseline FP32: F1=0.6623  size=17.3MB  1T=8.9ms  4T=5.4ms
MobileNetV3-L    baseline INT8: F1=0.4578  size=4.8MB  1T=13.5ms  4T=9.8ms


/tmp/ipykernel_58/361664559.py:51: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net, torch.randn(1, 3, 224, 224), f32, input_names=["input"], output_names=["logits"],


MobileNetV3-L    kd       FP32: F1=0.7082  size=17.3MB  1T=9.0ms  4T=5.4ms
MobileNetV3-L    kd       INT8: F1=0.5330  size=4.8MB  1T=13.3ms  4T=9.5ms


/tmp/ipykernel_58/361664559.py:51: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net, torch.randn(1, 3, 224, 224), f32, input_names=["input"], output_names=["logits"],


...

EfficientNet-B0  kd       FP32: F1=0.7161  size=16.5MB  1T=18.6ms  4T=11.0ms
EfficientNet-B0  kd       INT8: F1=0.1179  size=4.9MB  1T=27.9ms  4T=20.7ms


/tmp/ipykernel_58/361664559.py:51: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net, torch.randn(1, 3, 224, 224), f32, input_names=["input"], output_names=["logits"],


MobileNetV4-S    baseline FP32: F1=0.5765  size=10.4MB  1T=4.7ms  4T=2.9ms
MobileNetV4-S    baseline INT8: F1=0.4894  size=2.9MB  1T=5.1ms  4T=3.2ms


/tmp/ipykernel_58/361664559.py:51: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net, torch.randn(1, 3, 224, 224), f32, input_names=["input"], output_names=["logits"],


MobileNetV4-S    kd       FP32: F1=0.6563  size=10.4MB  1T=5.0ms  4T=3.0ms
MobileNetV4-S    kd       INT8: F1=0.5056  size=2.9MB  1T=5.3ms  4T=3.2ms

=== INT8 DEPLOYMENT (PlantWild P2 test, seed 0) ===
                          cpu_ms_1thread         size_MB        test_f1        int8_drop_pp speedup_1T
precision                          FP32    INT8    FP32   INT8    FP32   INT8                        
model           method                                                                               
EfficientNet-B0 baseline         18.612  27.822  16.461  4.923   0.658  0.110       -54.77      0.669
                kd               18.597  27.866  16.461  4.923   0.716  0.118       -59.82      0.667
MobileNetV3-L   baseline          8.904  13.484  17.251  4.787   0.662  0.458       -20.45      0.660
                kd                9.017  13.337  17.251  4.787   0.708  0.533       -17.52      0.676
MobileNetV4-S   baseline          4.664   5.128  10.401  2.899   0.576  0.489      

In [7]:
# ===== Step 8.2: INT8 recipe search on VALIDATION, then final test + latency =====
import os, subprocess, numpy as np, pandas as pd, matplotlib.pyplot as plt, onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantFormat, QuantType, CalibrationMethod
from onnxruntime.quantization.shape_inference import quant_pre_process
from sklearn.metrics import f1_score
flags = open("/proc/cpuinfo").read()
print("CPU:", [l.split(":")[1].strip() for l in flags.splitlines() if l.startswith("model name")][0])
print("VNNI:", "avx512_vnni" in flags or "avx_vnni" in flags, "| AVX512:", "avx512f" in flags)
va = np.where(df.p2_role.values == "val")[0]
def f1_on(path, idx):
    so = ort.SessionOptions(); so.intra_op_num_threads = 4
    s = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"]); n = s.get_inputs()[0].name
    pred = np.concatenate([s.run(None, {n: prep(idx[i:i+64])})[0].argmax(1) for i in range(0, len(idx), 64)])
    return f1_score(y_all[idx], pred, average="macro"), (pred == y_all[idx]).mean()
P99 = dict(calibrate_method=CalibrationMethod.Percentile, extra_options={"CalibPercentile": 99.99})
RECIPES = {
    "A_minmax":        dict(calibrate_method=CalibrationMethod.MinMax),
    "B_percentile":    dict(P99),
    "C_entropy":       dict(calibrate_method=CalibrationMethod.Entropy),
    "D_pct_reduce":    dict(P99, reduce_range=True),
    "E_pct_reduce_conv": dict(P99, reduce_range=True, op_types_to_quantize=["Conv", "Gemm", "MatMul"]),
}
def quantize(src, dst, recipe, fmt=QuantFormat.QDQ):
    kw = dict(recipe); kw["extra_options"] = dict(kw.get("extra_options", {}))
    quantize_static(src, dst, Calib("input"), quant_format=fmt, per_channel=True,
                    activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8, **kw)

# 1) pre-process all FP32 models
for short in MODELS:
    for meth in ["baseline", "kd"]:
        src, pre = f"{OUT}/{short}_{meth}_fp32.onnx", f"{OUT}/{short}_{meth}_pre.onnx"
        if os.path.exists(src) and not os.path.exists(pre): quant_pre_process(src, pre)
# 2) recipe search on KD models, validation only
search = []
for short, nice in MODELS.items():
    pre = f"{OUT}/{short}_kd_pre.onnx"; f32v, _ = f1_on(pre, va)
    search.append(dict(model=nice, recipe="FP32 (reference)", val_f1=f32v))
    for rn, rc in RECIPES.items():
        q = f"{OUT}/{short}_kd_{rn}.onnx"
        try:
            quantize(pre, q, rc); v, _ = f1_on(q, va)
        except Exception as e:
            print(f"  {nice} {rn} failed: {type(e).__name__}: {str(e)[:120]}"); v = np.nan
        search.append(dict(model=nice, recipe=rn, val_f1=v)); print(f"{nice:16s} {rn:18s} val F1={v:.4f}  (FP32 {f32v:.4f})", flush=True)
SR = pd.DataFrame(search); SR.to_csv(f"{FIG}/table_int8_recipe_search.csv", index=False)
best = SR[SR.recipe != "FP32 (reference)"].dropna().sort_values("val_f1").groupby("model").tail(1).set_index("model").recipe
print("\nBest recipe per model (chosen on validation):", best.to_dict())

# 3) final: best recipe on baseline + KD -> test F1, size, latency (QDQ and QOperator)
rows = []
for short, nice in MODELS.items():
    rn = best[nice]
    for meth in ["baseline", "kd"]:
        pre = f"{OUT}/{short}_{meth}_pre.onnx"
        variants = [("FP32", pre)]
        for fmt, tag in [(QuantFormat.QDQ, "INT8-QDQ"), (QuantFormat.QOperator, "INT8-QOp")]:
            q = f"{OUT}/{short}_{meth}_final_{tag}.onnx"
            try: quantize(pre, q, RECIPES[rn], fmt); variants.append((tag, q))
            except Exception as e: print(f"  {nice} {meth} {tag} failed: {type(e).__name__}")
        for prec, path in variants:
            f1, acc = f1_on(path, te)
            rows.append(dict(model=nice, method=meth, precision=prec, recipe=rn if prec != "FP32" else "-",
                             size_MB=os.path.getsize(path) / 1e6, test_f1=f1, test_acc=acc,
                             cpu_ms_1thread=latency(path, 1), cpu_ms_4threads=latency(path, 4)))
            r = rows[-1]; print(f"{nice:16s} {meth:8s} {prec:9s} F1={f1:.4f} size={r['size_MB']:.1f}MB "
                                f"1T={r['cpu_ms_1thread']:.1f}ms 4T={r['cpu_ms_4threads']:.1f}ms", flush=True)
D = pd.DataFrame(rows).round(4); D.to_csv(f"{FIG}/table_int8_deployment.csv", index=False)
base = D[D.precision == "FP32"].set_index(["model", "method"]).test_f1
D["drop_vs_fp32_pp"] = [round(100 * (r.test_f1 - base[(r.model, r.method)]), 2) for _, r in D.iterrows()]
pd.set_option("display.width", 220); print("\n=== FINAL INT8 DEPLOYMENT (PlantWild P2 test) ===\n",
      D[["model", "method", "precision", "recipe", "size_MB", "test_f1", "drop_vs_fp32_pp", "cpu_ms_1thread", "cpu_ms_4threads"]].to_string(index=False))
D.to_csv(f"{FIG}/table_int8_deployment.csv", index=False)

fig, ax = plt.subplots(figsize=(6.2, 3.8)); mk = {"FP32": "o", "INT8-QDQ": "^", "INT8-QOp": "v"}; col = {"baseline": "#8C8C8C", "kd": "#4C72B0"}
for _, r in D.iterrows():
    ax.scatter(r.cpu_ms_1thread, r.test_f1, marker=mk[r.precision], color=col[r.method], s=50)
    if r.precision == "FP32" and r.method == "kd": ax.annotate(r.model, (r.cpu_ms_1thread, r.test_f1), fontsize=7, xytext=(4, 3), textcoords="offset points")
for p, m in mk.items(): ax.scatter([], [], marker=m, color="gray", label=p)
for k, c in col.items(): ax.scatter([], [], marker="s", color=c, label=k)
ax.set_xlabel("CPU latency, batch 1, 1 thread (ms)"); ax.set_ylabel("Test macro-F1 (PlantWild P2)"); ax.legend(fontsize=7); ax.grid(alpha=.3)
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/fig8_1_int8_deployment.{e}", dpi=300, bbox_inches="tight")
plt.close(fig)
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG); log = log[log.file != "fig8_1_int8_deployment"]
pd.concat([log, pd.DataFrame([dict(file="fig8_1_int8_deployment", step="8",
    caption="Accuracy vs single-thread CPU latency: FP32 and INT8 (validation-selected post-training quantisation recipe), baseline vs KD.")])]).to_csv(LOG, index=False)
print("saved fig8_1_int8_deployment\nStep 8.2 DONE.")

CPU: Intel(R) Xeon(R) CPU @ 2.00GHz
VNNI: False | AVX512: True
MobileNetV3-L    A_minmax           val F1=0.4950  (FP32 0.6887)
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 162
Number of histogram bins : 2048
Percentile : (0.010000000000005116,99.99)
MobileNetV3-L    B_percentile       val F1=0.5823  (FP32 0.6887)
Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 162
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128
MobileNetV3-L    C_entropy          val F1=0.4950  (FP32 0.6887)
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 162
Number of histogram bins : 2048
Percentile : (0.010000000000005116,99.99)
MobileNetV3-L    D_pct_reduce       val F1=0.6334  (FP32 0.6887)
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 123
Number of histog

In [8]:
# ===== Step 8.3: mixed-precision INT8 (FP32 depthwise / first / last layers), chosen on validation =====
import onnx, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
def sensitive_nodes(path, with_ends):
    m = onnx.load(path); changed = False
    for i, n in enumerate(m.graph.node):
        if not n.name: n.name = f"node_{i}_{n.op_type}"; changed = True
    if changed: onnx.save(m, path)
    convs = [n for n in m.graph.node if n.op_type == "Conv"]
    dw = [n.name for n in convs if any(a.name == "group" and a.i > 1 for a in n.attribute)]
    ends = ([convs[0].name] + [n.name for n in m.graph.node if n.op_type in ("Gemm", "MatMul")]) if with_ends else []
    return sorted(set(dw + ends)), len(dw)
MIX = {"F_dw_fp32": False, "G_dw_ends_fp32": True}
search2 = []
for short, nice in MODELS.items():
    pre = f"{OUT}/{short}_kd_pre.onnx"
    for rn, ends in MIX.items():
        excl, ndw = sensitive_nodes(pre, ends)
        RECIPES[rn] = dict(RECIPES["D_pct_reduce"], nodes_to_exclude=excl)
        q = f"{OUT}/{short}_kd_{rn}.onnx"; quantize(pre, q, RECIPES[rn]); v, _ = f1_on(q, va)
        search2.append(dict(model=nice, recipe=rn, val_f1=v, fp32_nodes=len(excl)))
        print(f"{nice:16s} {rn:16s} val F1={v:.4f}  (FP32 layers kept: {len(excl)}, depthwise: {ndw})", flush=True)
SR = pd.concat([pd.read_csv(f"{FIG}/table_int8_recipe_search.csv"), pd.DataFrame(search2)], ignore_index=True)
SR.to_csv(f"{FIG}/table_int8_recipe_search.csv", index=False)
best = SR[SR.recipe != "FP32 (reference)"].dropna(subset=["val_f1"]).sort_values("val_f1").groupby("model").tail(1).set_index("model").recipe
print("\nBest recipe per model (validation):", best.to_dict())

rows = []
for short, nice in MODELS.items():
    rn = best[nice]
    for meth in ["baseline", "kd"]:
        pre = f"{OUT}/{short}_{meth}_pre.onnx"
        if rn in MIX:                                     # node names differ per file -> rebuild exclusion list
            RECIPES[rn] = dict(RECIPES["D_pct_reduce"], nodes_to_exclude=sensitive_nodes(pre, MIX[rn])[0])
        q = f"{OUT}/{short}_{meth}_best_int8.onnx"; quantize(pre, q, RECIPES[rn], QuantFormat.QOperator)
        for prec, path in [("FP32", pre), ("INT8", q)]:
            f1, acc = f1_on(path, te)
            rows.append(dict(model=nice, method=meth, precision=prec, recipe=rn if prec == "INT8" else "-",
                             size_MB=os.path.getsize(path) / 1e6, test_f1=f1, test_acc=acc,
                             cpu_ms_1thread=latency(path, 1), cpu_ms_4threads=latency(path, 4)))
D = pd.DataFrame(rows)
base = D[D.precision == "FP32"].set_index(["model", "method"]).test_f1
D["drop_vs_fp32_pp"] = [100 * (r.test_f1 - base[(r.model, r.method)]) for _, r in D.iterrows()]
D = D.round(4); D.to_csv(f"{FIG}/table_int8_deployment.csv", index=False)
pd.set_option("display.width", 220)
print("\n=== FINAL INT8 DEPLOYMENT (best validation recipe, PlantWild P2 test) ===\n",
      D[["model", "method", "precision", "recipe", "size_MB", "test_f1", "drop_vs_fp32_pp", "cpu_ms_1thread", "cpu_ms_4threads"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(6.2, 3.8)); mk = {"FP32": "o", "INT8": "^"}; col = {"baseline": "#8C8C8C", "kd": "#4C72B0"}
for _, r in D.iterrows():
    ax.scatter(r.size_MB, r.test_f1, marker=mk[r.precision], color=col[r.method], s=55)
    if r.method == "kd" and r.precision == "INT8": ax.annotate(r.model, (r.size_MB, r.test_f1), fontsize=7, xytext=(4, 3), textcoords="offset points")
for p, m in mk.items(): ax.scatter([], [], marker=m, color="gray", label=p)
for k, c in col.items(): ax.scatter([], [], marker="s", color=c, label=k)
ax.set_xlabel("Model file size (MB)"); ax.set_ylabel("Test macro-F1 (PlantWild P2)"); ax.legend(fontsize=7); ax.grid(alpha=.3)
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/fig8_1_int8_deployment.{e}", dpi=300, bbox_inches="tight")
plt.close(fig)
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG); log = log[log.file != "fig8_1_int8_deployment"]
pd.concat([log, pd.DataFrame([dict(file="fig8_1_int8_deployment", step="8",
    caption="Accuracy vs model size for FP32 and INT8 (validation-selected mixed-precision PTQ), baseline vs KD students.")])]).to_csv(LOG, index=False)
print("saved fig8_1_int8_deployment\nStep 8.3 DONE.")

Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 162
Number of histogram bins : 2048
Percentile : (0.010000000000005116,99.99)
MobileNetV3-L    F_dw_fp32        val F1=0.3656  (FP32 layers kept: 15, depthwise: 15)
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 162
Number of histogram bins : 2048
Percentile : (0.010000000000005116,99.99)
MobileNetV3-L    G_dw_ends_fp32   val F1=0.3688  (FP32 layers kept: 17, depthwise: 15)
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 240
Number of histogram bins : 2048
Percentile : (0.010000000000005116,99.99)
EfficientNet-B0  F_dw_fp32        val F1=0.0432  (FP32 layers kept: 16, depthwise: 16)
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 240
Number of histogram bins : 2048
Percentile : (0.010000000000005116,99.99)
EfficientNet-B0  G_dw_ends_fp32   val F1=0.0423

In [9]:
import os
os.system("cd /kaggle/working && rm -f step8.zip && zip -qr step8.zip figures feats/xd_manifest.csv feats/kd_targets "
          "students/results.jsonl students/logits students/curves students/ckpt/xd_* deploy/*_pre.onnx deploy/*_best_int8.onnx")
print(round(os.path.getsize("/kaggle/working/step8.zip")/1e6), "MB")

436 MB


In [10]:
import os
from IPython.display import FileLink
print(os.path.getsize("/kaggle/working/step8.zip")/1e6, "MB")
os.chdir("/kaggle/working"); FileLink("step8.zip")

436.1141 MB


/kaggle/working/step8.zip